# 🔥 Monitor do Fogo – Trajetória e Origem do Fogo (v1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mapbiomas/brazil-fire/blob/main/mapbiomas_fire_monitor/fire_origin/version_01/mapbiomas_fire_origin.ipynb)

Agrupa os focos de calor mensais em **eventos de fogo** (ST-DBSCAN), estima a ordem de propagação dentro de cada evento e cruza com as cicatrizes mensais do Monitor do Fogo, gerando um mapa de **trajetória** a 30 m e as **zonas e pontos de origem** do fogo.

| Etapa | O que faz | Onde roda | Saída |
|---|---|---|---|
| 1 | Exporta os focos mensais | GEE → GCS | `01_FOCOS_MENSAIS_ORIGINAL/*.csv` |
| 2 | ST-DBSCAN, herança de clusters e Lapsetime | Colab | `02_st-dbscan-output_v1/*.csv` |
| 2.1 | Ingestão dos focos clusterizados | GEE | tabelas `focos-st-dbscan-output` |
| 3 | Rasterização e interpolação | Colab | `03_STFireOrigins-output_v1/*.tif` |
| 3.1 | Ingestão dos rasters | GEE | `st_fire_origins_raster30m_v1` |
| 4 | Consolidação com a área queimada (5 bandas) | GEE | `fire_monitor_trajetory_v1` |

**Como rodar**
1. Rode as células de **importação** e **autenticação**.
2. Ajuste o período em **Parâmetros Globais → O que processar**.
3. Rode as etapas em ordem. As etapas 1, 2.1, 3.1 e 4 só **enviam tarefas**: espere terminarem no [Task Manager](https://code.earthengine.google.com/tasks) antes de rodar a próxima.

> ⚠️ A última célula **apaga todas as imagens de `fire_monitor_trajetory_v1`**. Não use "Executar tudo".

# Definição de Parâmetros Globais

In [ ]:
%pip install -q rasterio

import gc
import time
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory

import ee
import gcsfs
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from scipy.interpolate import LinearNDInterpolator
from scipy.ndimage import distance_transform_edt
from scipy.spatial import QhullError
from shapely.geometry import Point
from sklearn.cluster import DBSCAN

In [ ]:
ee.Authenticate()
ee.Initialize(project='mapbiomas-fire-485203')

In [ ]:
############################# Parâmetros Globais do Algoritmo #############################

# 🎨 Códigos ANSI para mensagens coloridas no terminal
C_BLUE, C_GREEN, C_YELLOW, C_RED, C_RESET = "\033[1;34m", "\033[1;32m", "\033[1;33m", "\033[1;31m", "\033[0m"

# ==========================================================================================
# 1) 🔄 O QUE PROCESSAR  ← altere aqui a cada execução (ano/mês inicial e final)
# ==========================================================================================
export_start_year, export_end_year = 2026, 2026
export_start_month, export_end_month = 10, 10

# ==========================================================================================
# 2) ⚙️ PARÂMETROS ESPAÇO-TEMPORAIS (ST-DBSCAN)
# ==========================================================================================
# EVENTO DE FOGO = grupo de focos próximos no espaço e no tempo.
#   Vizinhos: até pa_eps_spatial de distância e até pa_eps_temporal horas entre si.
#   Vizinhos se encadeiam, então um evento pode passar desses limites.
#   Focos isolados (CLUSTER_ID = -1) são descartados.

pa_eps_spatial = 0.02       # MÁXIMO: distância entre dois focos vizinhos, em graus (0,02° ≈ 2,2 km).
                            #   Maior → junta focos mais distantes (eventos maiores); menor → separa mais.
pa_eps_temporal = 48        # MÁXIMO: intervalo de tempo entre dois focos vizinhos, em horas.
                            #   Maior → junta focos mais espaçados no tempo; menor → separa mais.
pa_min_samples = 2          # MÍNIMO: nº de focos na vizinhança (contando o próprio) para formar um evento.
                            #   2 = basta um par de focos próximos; um foco isolado não é evento.
pa_date_field = 'DataHora'  # Nome do campo de data/hora padronizado (não alterar)
# Notas originais destes parâmetros: "versão 1 2km", "versão1 24h", "versão 5 focos min."

# ==========================================================================================
# 3) 🪣 PROJETO, BUCKET E ASSETS
# ==========================================================================================
VERSAO = 'v1'  # sufixo das saídas deste produto (pastas no GCS e coleções no GEE)

# Google Cloud Storage (GCS): pastas de cada etapa
gcs_bucket_name = 'mapbiomas-fire'
gcs_root_path = f'gs://{gcs_bucket_name}/monitor/product_fire_origin'
gcs_input_raw_path = f'{gcs_root_path}/01_FOCOS_MENSAIS_ORIGINAL'      # Etapa 1 (saída)
gcs_base_path = f'{gcs_root_path}/02_st-dbscan-output_{VERSAO}'         # Etapa 2 (saída)
gcs_output_path = f'{gcs_root_path}/03_STFireOrigins-output_{VERSAO}'   # Etapa 3 (saída)

# Google Earth Engine (GEE): entradas
gee_project_id = 'mapbiomas-fire-485203'
gee_asset_base_path = f"projects/{gee_project_id}/assets/DATABASE/monthly-focus-sul-america"  # focos mensais
burned_asset_col = "projects/mapbiomas-public/assets/brazil/fire/monitor/mapbiomas_fire_monthly_burned_v1"  # área queimada mensal
brasil_geom = ee.FeatureCollection("projects/ee-ipam/assets/IBGE/IBGE_Biomas").geometry()  # região de exportação

# Google Earth Engine (GEE): saídas
gee_target_collection = f"projects/earthengine-legacy/assets/projects/mapbiomas-workspace/FOGO/MONITORAMENTO/st_fire_origins_raster30m_{VERSAO}"  # Etapa 3.1
gee_target_consolidated_collection = f"projects/mapbiomas-workspace/FOGO/MONITORAMENTO/fire_monitor_trajetory_{VERSAO}"  # Etapa 4

# ==========================================================================================
# 4) 🗺️ RASTER GEOTIFF E INTERPOLAÇÃO (Etapa 3)
# ==========================================================================================
BANDAS = ["R_lapsetime", "system:time_start", "CLUSTER_ID"]
RESOLUCAO = 0.001             # tamanho do pixel, em graus (≈ 110 m)
NODATA = -9999                # valor de "sem dado"
DISTANCIA_BUSCA = 15          # raio de preenchimento ao redor dos focos, em pixels
TAMANHO_BLOCO = 2048          # tamanho do bloco de leitura/escrita, em pixels
TAMANHO_CHUNK_CSV = 250_000   # linhas do CSV lidas por vez
INDICE_R, INDICE_TEMPO, INDICE_CLUSTER = BANDAS.index("R_lapsetime"), BANDAS.index("system:time_start"), BANDAS.index("CLUSTER_ID")
COLUNAS_CSV = {"Longitude", "Latitude", "DataHora", *BANDAS}
PASTA_TRABALHO = Path("/content/focos_processamento")

# 🛠️ Etapa 1: Exportação de Focos de Calor do GEE para o GCS
Envia uma tarefa por mês para salvar os focos mensais (América do Sul, todos os satélites) em CSV no bucket e **espera as tarefas terminarem** antes de liberar a próxima etapa.

- Pula o mês se o asset de focos ainda não existir no GEE.
- `SOBRESCREVER_ETAPA1 = False` pula os meses que já têm CSV no bucket.
- Cada foco recebe um `foco_id` único (`AAAA_MM_<id do GEE>`).

In [ ]:
# 📢 Header de Contexto
print("======================================================================")
print("📦 ETAPA 1: EXPORTAÇÃO LOTE GEE ➔ GCS (Com Padronização de Data/Hora)")
print("======================================================================")

# ⚙️ Opções desta etapa
SOBRESCREVER_ETAPA1 = True  # True: exporta de novo mesmo se o CSV já existir no bucket
AGUARDAR_TAREFAS = True     # True: a célula só termina quando as tarefas acabarem
INTERVALO_CONSULTA = 30     # segundos entre cada consulta ao status das tarefas

# 🪣 Acesso ao GCS (para conferir CSVs já existentes)
fs = gcsfs.GCSFileSystem(token="google_default")

# 🛠️ Função para tratar colunas e extrair Coordenadas no GEE
def process_feature(feature, prefixo_id):
    # 1. Extrai Lat/Lon da geometria
    coords = feature.geometry().coordinates()
    lon = coords.get(0)
    lat = coords.get(1)

    # 2. Padroniza a coluna de data/hora para 'datahora'
    data_val = ee.Algorithms.If(
        feature.propertyNames().contains('datahora'),
        feature.get('datahora'),
        ee.Algorithms.If(
            feature.propertyNames().contains('data_hora_gmt'),
            feature.get('data_hora_gmt'),
            feature.get('data_hora') # Fallback para data_hora
        )
    )

    # 3. Define as propriedades padronizadas (+ ID único do foco: AAAA_MM_<id do GEE>)
    return feature.set({
        'foco_id': ee.String(prefixo_id).cat(feature.id()),
        'longitude': lon,
        'latitude': lat,
        'datahora': data_val
    })

# 🔄 Loop de Exportação
tarefas_enviadas = []

for y in range(export_start_year, export_end_year + 1):
    m_start = export_start_month if y == export_start_year else 1
    m_end = export_end_month if y == export_end_year else 12

    for m in range(m_start, m_end + 1):
        gee_asset_path = f"{gee_asset_base_path}/focus_{y}-{m:02d}"
        file_prefix = f"monitor/product_fire_origin/01_FOCOS_MENSAIS_ORIGINAL/focos_{y}_{m:02d}"
        csv_gcs = f"{gcs_bucket_name}/{file_prefix}.csv"

        try:
            # 0. Confere se o asset de focos do mês existe no GEE
            try:
                ee.data.getAsset(gee_asset_path)
            except ee.EEException:
                print(f"{C_YELLOW}⚠️ Asset não encontrado: {gee_asset_path}. Pulando {y}-{m:02d}...{C_RESET}")
                continue

            # 0.1 Pula se o CSV já existe e não é para sobrescrever
            if not SOBRESCREVER_ETAPA1 and fs.exists(csv_gcs):
                print(f"{C_YELLOW}⏭️ CSV já existe, pulando {y}-{m:02d}: gs://{csv_gcs}{C_RESET}")
                continue

            print(f"\n🔥 [START] Extraindo e exportando: {y}-{m:02d}...")

            # 1. Carrega o Asset e aplica o processamento
            prefixo_id = f"{y}_{m:02d}_"
            focos_fc = ee.FeatureCollection(gee_asset_path).map(lambda f: process_feature(f, prefixo_id))

            # 2. Disparo da Task com os seletores padronizados
            task = ee.batch.Export.table.toCloudStorage(
                collection=focos_fc,
                description=f"export_focos_{y}_{m:02d}",
                bucket=gcs_bucket_name,
                fileNamePrefix=file_prefix,
                fileFormat="CSV",
                selectors=[
                    "foco_id", "datahora", "latitude", "longitude",
                    "satelite", "pais", "estado", "municipio", "bioma",
                    "diasemchuva", "precipitacao", "riscofogo", "frp"
                ]
            )
            task.start()
            tarefas_enviadas.append((f"{y}-{m:02d}", task))

            print(f"🚀 [SUCCESS] Task {y}-{m:02d} enviada!")
            print(f"📄 Destino: gs://{csv_gcs}")

            time.sleep(0.5)

        except Exception as e:
            print(f"❌ Erro na exportação de {y}-{m:02d}: {str(e)}")

# ⏳ Aguarda as tarefas terminarem (evita que a Etapa 2 leia um CSV antigo)
if AGUARDAR_TAREFAS and tarefas_enviadas:
    print(f"\n⏳ Aguardando {len(tarefas_enviadas)} tarefa(s) terminarem...")
    pendentes = dict(tarefas_enviadas)
    falhas = []

    while pendentes:
        time.sleep(INTERVALO_CONSULTA)
        for nome, task in list(pendentes.items()):
            status = task.status()
            estado = status.get("state")
            if estado == "COMPLETED":
                print(f"{C_GREEN}✅ {nome}: concluída{C_RESET}")
                del pendentes[nome]
            elif estado in ("FAILED", "CANCELLED"):
                print(f"{C_RED}❌ {nome}: {estado} – {status.get('error_message', '')}{C_RESET}")
                falhas.append(nome)
                del pendentes[nome]
        if pendentes:
            print(f"   … em andamento: {', '.join(pendentes)}")

    if falhas:
        print(f"\n{C_RED}⚠️ Tarefas com falha: {', '.join(falhas)}. Não rode a Etapa 2 para esses meses.{C_RESET}")
    else:
        print(f"\n{C_GREEN}✅ ETAPA 1: todas as tarefas concluídas. Pode seguir para a Etapa 2.{C_RESET}")
elif not tarefas_enviadas:
    print(f"\n{C_YELLOW}ℹ️ Nenhuma tarefa enviada.{C_RESET}")

# 🛠️ Etapa 2: Clusterização ST-DBSCAN & Análise Espaço-Temporal
Identifica os complexos de fogo (CLUSTER_ID), mapeia a herança entre meses e calcula os indicadores de propagação e ciclo de vida (Lapsetime e R_lapsetime).

In [ ]:

def _read_csv_optimized(file_path, fs_instance, initial_dtypes=None, convert_categories=None):
    """
    Reads a CSV file from GCS with memory optimization.
    - Specifies dtypes to reduce memory footprint and avoid warnings.
    - Converts specified columns to 'category' dtype.
    - Attempts to downcast numeric types.
    """
    if not fs_instance.exists(file_path):
        return pd.DataFrame()

    print(f"📥 Carregando e otimizando: {file_path.split('/')[-1]}...")

    # Dicionário de dtypes padrão para evitar inferência e otimizar memória
    default_dtypes = {
        'datahora': str,
        'DataHora': str,
        'system:time_start': np.int64,
        'latitude': np.float32,
        'longitude': np.float32,
        'satelite': 'category',
        'pais': 'category',
        'estado': 'category',
        'municipio': str,
        'bioma': str,
        'CLUSTER_ID': np.int32,
        'CLUSTER_SIZE': np.int32,
        'Lapsetime': np.int32,
        'R_lapsetime': np.float32,
        'Anual_lapsetime': np.int32,
        'Anual_R_lapsetime': np.float32,
        'T': np.int8
    }

    if initial_dtypes:
        default_dtypes.update(initial_dtypes)

    df = pd.read_csv(file_path, dtype={k: v for k, v in default_dtypes.items()}, low_memory=False)

    # Post-processing para conversão de categoria e downcasting numérico
    for col in df.columns:
        # Lista de colunas críticas de data que NUNCA devem virar 'category'
        if col.lower() in ['datahora', 'system:time_start']:
            continue

        if df[col].dtype == 'object':
            # Converta para categoria se a coluna estiver na lista convert_categories
            # ou se tiver uma baixa cardinalidade
            if convert_categories and col in convert_categories:
                df[col] = df[col].astype('category')
            elif df[col].nunique() / len(df[col]) < 0.5: # Ex: menos de 50% de valores únicos
                try:
                    df[col] = df[col].astype('category')
                except Exception:
                    pass

        elif pd.api.types.is_numeric_dtype(df[col]):
            # Downcast números para tipos de menor memória
            try:
                if pd.api.types.is_integer_dtype(df[col]):
                    df[col] = pd.to_numeric(df[col], downcast='integer')
                elif pd.api.types.is_float_dtype(df[col]):
                    df[col] = pd.to_numeric(df[col], downcast='float')
            except ValueError:
                pass

    gc.collect() # Força a coleta de lixo para liberar memória
    print(f"  Memória para {file_path.split('/')[-1]}: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    return df

In [ ]:
# 📢 Header de Contexto
print("======================================================================")
print("🔥 ETAPA 2: PROCESSAMENTO ST-DBSCAN & CONSOLIDAÇÃO RETROATIVA (GCS)")
print("======================================================================")

# 🪣 Inicialização do sistema de arquivos GCS
fs = gcsfs.GCSFileSystem(token="google_default")

# 🛠️ Função ST-DBSCAN Espaço-Temporal
def run_st_dbscan(gdf, eps_spatial, eps_temporal, min_samples, date_field):
    """Executa clusterização ST-DBSCAN vetorizada normalizando tempo e espaço."""
    coords = gdf[['Longitude', 'Latitude']].values if {'Longitude', 'Latitude'}.issubset(gdf.columns) else np.array([[p.x, p.y] for p in gdf.geometry])

    dates = pd.to_datetime(gdf[date_field], errors='coerce') if gdf[date_field].dtype == 'object' else gdf[date_field]
    epoch = pd.Timestamp('1970-01-01')
    time_values = ((dates - epoch).dt.total_seconds() / (24 * 3600)).fillna(0).values.reshape(-1, 1)

    # Normalização temporal para alinhar com a escala espacial (graus)
    eps_temporal_days = eps_temporal / 24.0
    time_scale_factor = eps_spatial / eps_temporal_days if eps_temporal_days > 0 else 1

    features = np.hstack([coords, time_values * time_scale_factor])
    return DBSCAN(eps=eps_spatial, min_samples=min_samples, metric='euclidean', n_jobs=-1).fit_predict(features)


# 🔄 Loop Principal de Execução em Lote por Anos e Meses
for ano in range(export_start_year, export_end_year + 1):
    m_start = export_start_month if ano == export_start_year else 1
    m_end = export_end_month if ano == export_end_year else 12

    for mes in range(m_start, m_end + 1):
        print(f"\n======================================================================")
        print(f"📌 Processando ST-DBSCAN para o período: {ano}-{mes:02d}")
        print(f"======================================================================")

        prev_ano, prev_mes = (ano - 1, 12) if mes == 1 else (ano, mes - 1)

        # Caminhos unificados no GCS
        curr_raw_csv = f"{gcs_input_raw_path}/focos_{ano}_{mes:02d}.csv"
        prev_dbscan_csv = f"{gcs_base_path}/DBSCAN_{prev_ano}_{prev_mes:02d}.csv"
        output_dbscan_csv = f"{gcs_base_path}/DBSCAN_{ano}_{mes:02d}.csv"

        # 1. Carregamento dos dados brutos do mês atual do GCS
        if not fs.exists(curr_raw_csv):
            print(f"⚠️ Arquivo bruto não encontrado no GCS: {curr_raw_csv}. Pulando mês...")
            continue

        print(f"📥 Carregando focos brutos: gs://{gcs_bucket_name}/.../focos_{ano}_{mes:02d}.csv")
        focos_df = _read_csv_optimized(curr_raw_csv, fs) # Usando função otimizada
        focos_df['T'] = 0  # Marcador do mês atual

        # Tratamento flexível da coluna de data/hora (Ajuste para o Asset do GEE)
        if 'datahora' in focos_df.columns:
            focos_df['DataHora'] = pd.to_datetime(focos_df['datahora'], errors='coerce')
        elif 'DataHora' in focos_df.columns:
            focos_df['DataHora'] = pd.to_datetime(focos_df['DataHora'], errors='coerce')
        elif 'system:time_start' in focos_df.columns:
            focos_df['DataHora'] = pd.to_datetime(focos_df['system:time_start'], unit='ms')

        # Normalização de nomes de colunas de coordenadas para Maiúsculas
        if 'latitude' in focos_df.columns and 'longitude' in focos_df.columns:
            focos_df = focos_df.rename(columns={'latitude': 'Latitude', 'longitude': 'Longitude'})

        # 2. Carregamento opcional do mês anterior para continuidade de clusters
        if fs.exists(prev_dbscan_csv):
            prev_focos_df = _read_csv_optimized(prev_dbscan_csv, fs) # Usando função otimizada
            prev_focos_df['T'] = -1  # Marcador de dados do mês anterior
            prev_focos_df['DataHora'] = pd.to_datetime(prev_focos_df['DataHora'], errors='coerce')
            print(f"📂 Mês anterior carregado ({len(prev_focos_df)} registros para continuidade).")
        else:
            prev_focos_df = pd.DataFrame()
            print("ℹ️ Mês anterior não encontrado no GCS. Processando como início de ciclo.")

        # Concatenar para análise contínua e reiniciar o índice explicitamente após o dropna para evitar KeyErrors no loop .loc[i]
        combined_df = pd.concat([focos_df, prev_focos_df], ignore_index=True).dropna(subset=['Latitude', 'Longitude', 'DataHora']).reset_index(drop=True)

        # Limpeza de memória imediata
        del focos_df, prev_focos_df
        gc.collect()

        # Trava de segurança: se o mês estiver sem pontos, pula o processamento
        if len(combined_df) == 0:
            print(f"⚠️ NENHUM REGISTRO VÁLIDO ENCONTRADO EM {ano}-{mes:02d}. Pulando ST-DBSCAN...")
            continue

        geometry = [Point(lon, lat) for lon, lat in zip(combined_df['Longitude'], combined_df['Latitude'])]
        focos_gdf = gpd.GeoDataFrame(combined_df, geometry=geometry)

        # 3. Aplicação do ST-DBSCAN
        print("🧮 Executando ST-DBSCAN espaço-temporal...")
        raw_labels = run_st_dbscan(focos_gdf, pa_eps_spatial, pa_eps_temporal, pa_min_samples, pa_date_field)

        # 4. Mapeamento e Herança de CLUSTER_IDs
        adjusted_labels = []
        cluster_mapping = {}

        # Precisamos achar o max CLUSTER_ID dos pontos com T == -1
        max_prev_id = focos_gdf[focos_gdf['T'] == -1]['CLUSTER_ID'].max() if 'CLUSTER_ID' in focos_gdf.columns else 0
        next_cluster_id = (max_prev_id + 1) if pd.notna(max_prev_id) else 0

        # Mapeia os pontos antigos para herdar os IDs existentes
        for i, label in enumerate(raw_labels):
            if focos_gdf.loc[i, 'T'] == -1 and label != -1 and 'CLUSTER_ID' in focos_gdf.columns:
                orig_id = focos_gdf.loc[i, 'CLUSTER_ID']
                if label not in cluster_mapping:
                    cluster_mapping[label] = orig_id

        # Atribui os IDs finais para todos os pontos
        for i, label in enumerate(raw_labels):
            if focos_gdf.loc[i, 'T'] == -1:
                adjusted_labels.append(focos_gdf.loc[i, 'CLUSTER_ID'] if 'CLUSTER_ID' in focos_gdf.columns else label)
            else:
                if label == -1:
                    adjusted_labels.append(-1)
                else:
                    if label not in cluster_mapping:
                        cluster_mapping[label] = next_cluster_id
                        next_cluster_id += 1
                    adjusted_labels.append(cluster_mapping[label])

        focos_gdf['CLUSTER_ID'] = adjusted_labels
        focos_gdf = focos_gdf[focos_gdf['CLUSTER_ID'] >= 0].reset_index(drop=True)

        # Limpeza pesada antes da consolidação
        del combined_df, geometry, raw_labels, adjusted_labels, cluster_mapping
        gc.collect()

        # Separar apenas os dados do mês atual para exportação direta
        focos_curr = focos_gdf[focos_gdf['T'] != -1].copy().drop(columns=['T'])

        # 5. Cálculo dos Indicadores Temporais (Lapsetime e R_lapsetime)
        print("⏱️ Calculando Lapsetime e R_lapsetime (Mensal e Anual)...")
        min_dt = focos_curr.groupby('CLUSTER_ID')['DataHora'].transform('min')
        focos_curr['Lapsetime'] = ((focos_curr['DataHora'] - min_dt).dt.total_seconds() / 3600).astype(int)
        max_lapse = focos_curr.groupby('CLUSTER_ID')['Lapsetime'].transform('max')
        focos_curr['R_lapsetime'] = np.where(max_lapse > 0, focos_curr['Lapsetime'] / max_lapse, 0.0)

        # Indicadores de Lapsetime Anual
        focos_curr['ano_ref'] = focos_curr['DataHora'].dt.year
        hours_in_year = focos_curr['ano_ref'].apply(lambda y: (366 if ((y % 4 == 0 and y % 100 != 0) or (y % 400 == 0)) else 365) * 24)
        start_of_year = pd.to_datetime(focos_curr['ano_ref'].astype(str) + '-01-01 00:00:00')

        focos_curr['Anual_lapsetime'] = ((focos_curr['DataHora'] - start_of_year).dt.total_seconds() / 3600).astype(int)
        focos_curr['Anual_R_lapsetime'] = focos_curr['Anual_lapsetime'] / hours_in_year
        focos_curr['system:time_start'] = focos_curr['DataHora'].astype('int64') // 10**6

        # Limpeza e Exportação do arquivo do mês atual para o GCS
        export_df = focos_curr.drop(columns=['geometry', 'ano_ref'], errors='ignore')
        export_df.to_csv(output_dbscan_csv, index=False)
        print(f"{C_GREEN}🚀 [SUCCESS] Mês {ano}_{mes:02d} processado e salvo no GCS!{C_RESET}")

        # 6. Consolidação Retroativa (Atualização dos 6 meses anteriores no GCS)
        cluster_triggers = focos_gdf[focos_gdf['T'] == -1]['CLUSTER_ID'].unique()

        # Liberar a memória dos focos atuais, só precisamos do export_df para retroativa
        del focos_gdf, focos_curr, min_dt, max_lapse, hours_in_year, start_of_year
        gc.collect()

        if len(cluster_triggers) > 0:
            print("🔄 Iniciando atualização retroativa dos arquivos anteriores no GCS...")

            retro_dfs = [export_df[export_df['CLUSTER_ID'].isin(cluster_triggers)]]
            curr_m, curr_y = mes, ano

            for _ in range(6):
                curr_m -= 1
                if curr_m == 0:
                    curr_m, curr_y = 12, curr_y - 1

                path_check = f"{gcs_base_path}/DBSCAN_{curr_y}_{curr_m:02d}.csv"
                if fs.exists(path_check):
                    df_hist = _read_csv_optimized(path_check, fs) # Usando função otimizada
                    if any(c in df_hist['CLUSTER_ID'].values for c in cluster_triggers):
                        retro_dfs.append(df_hist)
                    else:
                        del df_hist # Liberar imediatamente se não for usar
                        gc.collect()

            # Recalcula os tempos unificados no DBASE consolidado
            if len(retro_dfs) > 1:
                consolidated = pd.concat(retro_dfs, ignore_index=True)
                # Liberar lista de dfs da memória assim que o concat terminar!
                del retro_dfs
                gc.collect()

                consolidated['DataHora'] = pd.to_datetime(consolidated['DataHora'], errors='coerce')

                min_dt_retro = consolidated.groupby('CLUSTER_ID')['DataHora'].transform('min')
                consolidated['Lapsetime'] = ((consolidated['DataHora'] - min_dt_retro).dt.total_seconds() / 3600).astype(int)
                max_lapse_retro = consolidated.groupby('CLUSTER_ID')['Lapsetime'].transform('max')
                consolidated['R_lapsetime'] = np.where(max_lapse_retro > 0, consolidated['Lapsetime'] / max_lapse_retro, 0.0)

                consolidated['merge_key'] = (consolidated['Latitude'].astype(str) + '_'
                                            + consolidated['Longitude'].astype(str) + '_'
                                            + consolidated['DataHora'].astype(str))

                consolidated['mes_ano'] = consolidated['DataHora'].dt.to_period('M')

                # Atualiza os CSVs no GCS com os novos tempos recalculados
                for period, group in consolidated.groupby('mes_ano'):
                    file_retro = f"{gcs_base_path}/DBSCAN_{period.year}_{period.month:02d}.csv"
                    if fs.exists(file_retro):
                        target_df = _read_csv_optimized(file_retro, fs)
                        target_df['DataHora'] = pd.to_datetime(target_df['DataHora'], errors='coerce')
                        target_df['merge_key'] = (target_df['Latitude'].astype(str) + '_'
                                                 + target_df['Longitude'].astype(str) + '_'
                                                 + target_df['DataHora'].astype(str))

                        # PREVENÇÃO DE EXPLOSÃO DE MEMÓRIA (OOM)
                        # Remove as chaves duplicadas do grupo para evitar o cruzamento cartesiano
                        group_dedup = group[['merge_key', 'Lapsetime', 'R_lapsetime']].drop_duplicates(subset=['merge_key'])

                        merged = target_df.merge(group_dedup, on='merge_key', how='left', suffixes=('', '_new'))

                        if 'Lapsetime_new' in merged.columns:
                            merged['Lapsetime'] = merged['Lapsetime_new'].fillna(merged['Lapsetime']).astype(int)
                            merged['R_lapsetime'] = merged['R_lapsetime_new'].fillna(merged['R_lapsetime'])
                            merged = merged.drop(columns=['Lapsetime_new', 'R_lapsetime_new', 'merge_key'])

                        merged.to_csv(file_retro, index=False)
                        print(f"  ✓ {file_retro.split('/')[-1]} atualizado com sucesso no GCS.")

                        # MUITO IMPORTANTE: Limpar RAM dentro do loop interno
                        del target_df, merged, group_dedup
                        gc.collect()

                del consolidated
                gc.collect()

print(f"\n{C_GREEN}✅ Processamento ST-DBSCAN de todos os meses concluído com sucesso!{C_RESET}")

# 🛠️ Etapa 2.1: Ingestão de focos de calor clusterizados no Google Earth Engine

In [ ]:
print("=" * 70)
print("📤 ETAPA 3: INGESTÃO DE ASSETS NO EARTH ENGINE")
print("=" * 70)

# 📁 Pasta no GEE
gee_target_folder = ("projects/earthengine-legacy/assets/projects/mapbiomas-workspace/FOGO/MONITORAMENTO/focos-st-dbscan-output")
print(f"📁 Pasta destino: {gee_target_folder}")

# 🔄 Ingestão GCS ➔ GEE
for ano in range(export_start_year, export_end_year + 1):
    m_start = export_start_month if ano == export_start_year else 1
    m_end = export_end_month if ano == export_end_year else 12

    for mes in range(m_start, m_end + 1):
        nome = f"DBSCAN_{ano}_{mes:02d}"
        gcs_csv_path = f"gs://{gcs_bucket_name}/monitor/product_fire_origin/02_st-dbscan-output/{nome}.csv"
        asset_id = f"{gee_target_folder}/{nome}"

        if not fs.exists(gcs_csv_path):
            print(f"⚠️ CSV não encontrado: {nome}")
            continue

        try:
            manifest = {
                "name": asset_id,
                "sources": [{
                    "uris": [gcs_csv_path],
                    "crs": "EPSG:4326"
                }]
            }

            task = ee.data.startTableIngestion(None, manifest)
            print(f"🚀 {nome} | Task: {task['id']}")
            print(f"📍 Destino: {asset_id}")

            time.sleep(0.5)

        except Exception as e:
            print(f"❌ Erro em {nome}: {e}")

print("\n✅ Processo de envio finalizado!")

# 🛠️ Etapa 3: Rasterização focos de calor clusterizados

In [ ]:
# 📂 Função utilitária para gerar a URI do CSV no GCS a partir do ano e mês
def uri_csv(ano: int, mes: int) -> str:
    return f"{gcs_base_path}/DBSCAN_{ano}_{mes:02d}.csv"

# 📖 Lê chunks de CSV do GCS selecionando apenas as colunas necessárias
def ler_chunks(uri: str, colunas: set[str]):
    return pd.read_csv(uri, usecols=lambda c: c in colunas, chunksize=TAMANHO_CHUNK_CSV, storage_options={"token": "google_default"})

# 🧹 Filtra linhas válidas removendo registros sem coordenadas válidas
def linhas_validas(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk = chunk.dropna(subset=["Longitude", "Latitude"])
    return chunk if chunk.empty else chunk.loc[np.isfinite(chunk[["Longitude", "Latitude"]].to_numpy(dtype="float64")).all(axis=1)]

# 📐 Calcula o bounding box (extensão espacial) agregada de todos os CSVs
def calcular_extensao(arquivos_csv: list[str]) -> tuple[float, float, float, float]:
    minx, miny, maxx, maxy, total = float("inf"), float("inf"), float("-inf"), float("-inf"), 0
    for arquivo in arquivos_csv:
        print(f"Lendo extensão: {arquivo}")
        for chunk in ler_chunks(arquivo, {"Longitude", "Latitude"}):
            v = linhas_validas(chunk)
            if v.empty: continue
            coords = v[["Longitude", "Latitude"]].to_numpy(dtype="float64")
            minx, maxx = min(minx, float(coords[:, 0].min())), max(maxx, float(coords[:, 0].max()))
            miny, maxy = min(miny, float(coords[:, 1].min())), max(maxy, float(coords[:, 1].max()))
            total += len(v)
    if total == 0: raise ValueError("Nenhum foco válido encontrado nos CSVs.")
    return minx, miny, maxx, maxy

# 🗺️ Cria o perfil GeoTIFF e calcula as dimensões em pixels do raster
def criar_perfil(minx: float, miny: float, maxx: float, maxy: float) -> tuple[dict, int, int]:
    largura, altura = int(np.ceil((maxx - minx) / RESOLUCAO)) + 1, int(np.ceil((maxy - miny) / RESOLUCAO)) + 1
    perfil = {
        "driver": "GTiff", "height": altura, "width": largura, "count": len(BANDAS),
        "dtype": "float64", "crs": "EPSG:4326", "transform": from_origin(minx, maxy, RESOLUCAO, RESOLUCAO),
        "nodata": NODATA, "tiled": True, "blockxsize": 512, "blockysize": 512, "compress": "deflate", "BIGTIFF": "YES"
    }
    return perfil, largura, altura

# 🔢 Extrai os valores das bandas e calcula o timestamp em milissegundos se necessário
def valores_das_bandas(chunk: pd.DataFrame) -> np.ndarray:
    if "system:time_start" not in chunk.columns:
        chunk["system:time_start"] = (chunk["DataHora"] - pd.Timestamp("1970-01-01")) // pd.Timedelta(milliseconds=1)
    valores = chunk[BANDAS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype="float64")
    valores[~np.isfinite(valores)] = NODATA
    return valores

# 🧱 Rasteriza os focos de calor pontuais em janelas do raster GeoTIFF
def rasterizar_chunks(arquivos_csv: list[str], caminho_raster: Path, perfil: dict, largura: int, altura: int, minx: float, maxy: float) -> None:
    colunas_bloco = (largura + TAMANHO_BLOCO - 1) // TAMANHO_BLOCO
    with rasterio.open(caminho_raster, "w+", **perfil) as destino:
        for l_ini in range(0, altura, TAMANHO_BLOCO):
            for c_ini in range(0, largura, TAMANHO_BLOCO):
                j = Window(c_ini, l_ini, min(TAMANHO_BLOCO, largura - c_ini), min(TAMANHO_BLOCO, altura - l_ini))
                destino.write(np.full((len(BANDAS), int(j.height), int(j.width)), NODATA, dtype="float64"), window=j)

        for arquivo in arquivos_csv:
            print(f"Rasterizando: {arquivo}")
            for chunk in ler_chunks(arquivo, COLUNAS_CSV):
                chunk["DataHora"] = pd.to_datetime(chunk["DataHora"])
                c_val = linhas_validas(chunk)
                if c_val.empty: continue
                lons, lats = c_val["Longitude"].to_numpy("float64"), c_val["Latitude"].to_numpy("float64")
                l_flt, c_flt = (maxy - lats) / RESOLUCAO, (lons - minx) / RESOLUCAO
                l_arr, c_arr = np.rint(l_flt), np.rint(c_flt)
                l_flt = np.where(np.isclose(l_flt, l_arr, rtol=0, atol=1e-9), l_arr, l_flt)
                c_flt = np.where(np.isclose(c_flt, c_arr, rtol=0, atol=1e-9), c_arr, c_flt)
                linhas, colunas = np.floor(l_flt).astype("int64"), np.floor(c_flt).astype("int64")
                valores = valores_das_bandas(c_val)

                id_blocos = (linhas // TAMANHO_BLOCO) * colunas_bloco + colunas // TAMANHO_BLOCO
                ordem = np.argsort(id_blocos, kind="stable")
                id_blocos, linhas, colunas, valores = id_blocos[ordem], linhas[ordem], colunas[ordem], valores[ordem]
                blocos, inicios = np.unique(id_blocos, return_index=True)
                fins = np.append(inicios[1:], len(id_blocos))

                for bloco_id, inicio, fim in zip(blocos, inicios, fins):
                    l_ini, c_ini = int(bloco_id // colunas_bloco) * TAMANHO_BLOCO, int(bloco_id % colunas_bloco) * TAMANHO_BLOCO
                    j = Window(c_ini, l_ini, min(TAMANHO_BLOCO, largura - c_ini), min(TAMANHO_BLOCO, altura - l_ini))
                    dados = destino.read(window=j)
                    loc_l, loc_c = linhas[inicio:fim] - l_ini, colunas[inicio:fim] - c_ini
                    idx_px = loc_l * int(j.width) + loc_c
                    _, idx_rev = np.unique(idx_px[::-1], return_index=True)
                    idx_ult = np.sort(len(idx_px) - 1 - idx_rev)
                    dados[:, loc_l[idx_ult], loc_c[idx_ult]] = valores[inicio:fim][idx_ult].T
                    destino.write(dados, window=j)

# 📊 Auxiliar para agrupar índices de array pelo ID do cluster
def agrupar_por_cluster(clusters: np.ndarray):
    ordem = np.argsort(clusters, kind="stable")
    ordenados = clusters[ordem]
    ids, inicios = np.unique(ordenados, return_index=True)
    return {ident: ordem[i:f] for ident, i, f in zip(ids, inicios, np.append(inicios[1:], len(ordenados)))}

# 🔮 Executa a interpolação espacial espacialmente dentro de cada cluster de fogo
def preencher_por_cluster(dados: np.ndarray) -> np.ndarray:
    val_r, val_tempo, clusters = dados[INDICE_R], dados[INDICE_TEMPO], dados[INDICE_CLUSTER]
    sementes = clusters != NODATA
    saida = np.full_like(dados, NODATA)
    if not sementes.any(): return saida

    distancias, (lin_prox, col_prox) = distance_transform_edt(~sementes, return_indices=True)
    alcance = distancias <= DISTANCIA_BUSCA
    cluster_px = clusters[lin_prox, col_prox]
    saida[INDICE_CLUSTER] = np.where(alcance, cluster_px, NODATA)
    saida[INDICE_TEMPO] = np.where(alcance, val_tempo[lin_prox, col_prox], NODATA)
    saida[INDICE_R] = np.where(alcance, val_r[lin_prox, col_prox], NODATA)

    l_alvo, c_alvo = np.nonzero(alcance)
    l_sem, c_sem = np.nonzero(sementes)
    alvos, grp_sem = agrupar_por_cluster(cluster_px[l_alvo, c_alvo]), agrupar_por_cluster(clusters[l_sem, c_sem])

    for c_id, idx_sem in grp_sem.items():
        if len(idx_sem) < 3 or c_id not in alvos: continue
        ls, cs = l_sem[idx_sem], c_sem[idx_sem]
        try:
            interp = LinearNDInterpolator(np.column_stack([cs, ls]), val_r[ls, cs])
        except QhullError: continue
        la, ca = l_alvo[alvos[c_id]], c_alvo[alvos[c_id]]
        res = interp(np.column_stack([ca, la]))
        val = np.isfinite(res)
        saida[INDICE_R][la[val], ca[val]] = res[val]
    return saida

# 🔄 Aplica a interpolação em blocos com margem espacial de busca
def preencher_nodata(caminho_raster: Path, caminho_preenchido: Path, perfil: dict, largura: int, altura: int) -> None:
    with rasterio.open(caminho_raster) as origem, rasterio.open(caminho_preenchido, "w", **perfil) as destino:
        for l_ini in range(0, altura, TAMANHO_BLOCO):
            for c_ini in range(0, largura, TAMANHO_BLOCO):
                j = Window(c_ini, l_ini, min(TAMANHO_BLOCO, largura - c_ini), min(TAMANHO_BLOCO, altura - l_ini))
                m_l, m_c = min(DISTANCIA_BUSCA, l_ini), min(DISTANCIA_BUSCA, c_ini)
                j_amp = Window(c_ini - m_c, l_ini - m_l, min(j.width + m_c + DISTANCIA_BUSCA, largura - c_ini + m_c), min(j.height + m_l + DISTANCIA_BUSCA, altura - l_ini + m_l))
                dados = preencher_por_cluster(origem.read(window=j_amp))
                destino.write(dados[:, m_l : m_l + int(j.height), m_c : m_c + int(j.width)], window=j)
        destino.descriptions = tuple(BANDAS)

# ☁️ Realiza o upload do GeoTIFF processado diretamente para o bucket GCS
def enviar_para_gcs(caminho_tif: Path) -> None:
    caminho_obj = f"{gcs_output_path}/{caminho_tif.name}"
    if fs.exists(caminho_obj):
        raise FileExistsError(f"O TIFF já existe no bucket: gs://{caminho_obj}")
    print(f"Enviando para gs://{caminho_obj}")
    fs.put(str(caminho_tif), caminho_obj)

# 🚀 Função principal que gerencia o loop de execução por mês/ano
def main() -> None:
    # Inicializa a conexão do sistema de arquivos GCS
    global fs
    fs = gcsfs.GCSFileSystem(token="google_default")

    PASTA_TRABALHO.mkdir(parents=True, exist_ok=True)

    for ano in range(export_start_year, export_end_year + 1):
        m_ini = export_start_month if ano == export_start_year else 1
        m_fim = export_end_month if ano == export_end_year else 12

        for mes in range(m_ini, m_fim + 1):
            nome_raster = f"STFIreOrigins_{ano}_{mes:02d}.tif"
            caminho_gcs_dest = f"{gcs_output_path}/{nome_raster}"

            print(f"\n{C_BLUE}==========================================")
            print(f"⏳ PROCESSANDO RASTER: {ano}_{mes:02d}")
            print(f"=========================================={C_RESET}")

            if fs.exists(caminho_gcs_dest):
                print(f"{C_YELLOW}⚠️ Pula {ano}_{mes:02d}: arquivo já existe no GCS.{C_RESET}")
                continue

            uri_input = uri_csv(ano, mes)
            if not fs.exists(uri_input.removeprefix("gs://")):
                print(f"{C_RED}❌ Pula {ano}_{mes:02d}: CSV não encontrado ({uri_input}).{C_RESET}")
                continue

            caminho_out_local = PASTA_TRABALHO / nome_raster
            arquivos_csv = [uri_input]

            minx, miny, maxx, maxy = calcular_extensao(arquivos_csv)
            perfil, largura, altura = criar_perfil(minx, miny, maxx, maxy)

            with TemporaryDirectory(dir=PASTA_TRABALHO) as tmp_dir:
                caminho_inter = Path(tmp_dir) / "focos_raster_intermediario.tif"
                rasterizar_chunks(arquivos_csv, caminho_inter, perfil, largura, altura, minx, maxy)
                preencher_nodata(caminho_inter, caminho_out_local, perfil, largura, altura)
                enviar_para_gcs(caminho_out_local)
                print(f"{C_GREEN}✅ Upload concluído com sucesso para {ano}_{mes:02d}!{C_RESET}")

if __name__ == "__main__":
    main()

# 🛠️ Etapa 3.1: Ingestão de focos de calor clusterizados e rasterizados no Google Earth Engine

In [ ]:
fs = gcsfs.GCSFileSystem(token="google_default")

print("=" * 70)
print("📤 ETAPA 3: INGESTÃO DINÂMICA DE RASTERS (TIFF) NO EARTH ENGINE")
print("=" * 70)

print(f"📁 Pasta destino (ImageCollection): {gee_target_collection}")

# Nomes das bandas, adaptados às regras do Earth Engine
NOMAS_BANDAS = ["R_lapsetime", "system_time_start", "CLUSTER_ID"]

for ano in range(export_start_year, export_end_year + 1):
    m_start = export_start_month if ano == export_start_year else 1
    m_end = export_end_month if ano == export_end_year else 12

    for mes in range(m_start, m_end + 1):
        nome_gcs = f"STFIreOrigins_{ano}_{mes:02d}"
        nome_asset = f"{ano}_{mes:02d}"

        gcs_tif_path = f"{gcs_output_path}/{nome_gcs}.tif"
        asset_id = f"{gee_target_collection}/{nome_asset}"

        # Verifica se o GeoTIFF existe no GCS
        if not fs.exists(gcs_tif_path.replace("gs://", "")):
            print(f"⚠️ GeoTIFF não encontrado no GCS: {nome_gcs}.tif")
            continue

        try:
            # Lê a quantidade de bandas do GeoTIFF
            with rasterio.open(gcs_tif_path) as src:
                num_bandas = src.count

            # Verifica se a quantidade de bandas está correta
            if num_bandas != len(NOMAS_BANDAS):
                print(
                    f"⚠️ {nome_gcs}: esperado {len(NOMAS_BANDAS)} bandas, "
                    f"encontradas {num_bandas}. Pulando."
                )
                continue

            # Mapeia as bandas na ordem original
            bandas_manifest = [
                {
                    "id": b,
                    "tileset_id": "tileset_1",
                    "tileset_band_index": i
                }
                for i, b in enumerate(NOMAS_BANDAS)
            ]

            manifest = {
                "name": asset_id,
                "tilesets": [
                    {
                        "id": "tileset_1",
                        "sources": [{"uris": [gcs_tif_path]}]
                    }
                ],
                "bands": bandas_manifest,
                "start_time": datetime(
                    ano, mes, 1, tzinfo=timezone.utc
                ).isoformat().replace("+00:00", "Z"),
                "properties": {
                    "year": ano,
                    "month": mes,
                    "total_bands": num_bandas
                }
            }

            task_id = ee.data.newTaskId()[0]
            ee.data.startIngestion(task_id, manifest)

            print(
                f"🚀 {nome_gcs} ({num_bandas} bandas) "
                f"➔ {nome_asset} | Task ID: {task_id}"
            )

            time.sleep(0.5)

        except Exception as e:
            print(f"❌ Erro ao enviar {nome_gcs}: {e}")

print("\n✅ Solicitações de ingestão enviadas!")

#🛠️ Etapa 4: Consolidação 5 Bandas (Trajetória + Origem) no Monitor do Fogo & Exportação para ImageCollection

In [ ]:
"""
==============================================================================
🔥 PASSO 04 — CONSOLIDAÇÃO E EXPORTAÇÃO DE TRAJETÓRIAS E ORIGENS DO FOGO (V1)
==============================================================================
Consolida as cicatrizes mensais do Monitor do Fogo com os rasters ST-DBSCAN,
utilizando busca temporal em cascata (±3 meses).

Prioridade temporal: M, M-1, M+1, M-2, M+2, M-3, M+3.

BANDAS FINAIS:
  1. fire_relative_propagation_lapestime : R_lapsetime (0 a 1; 2 = fallback)
  2. fire_cluster_id                     : CLUSTER_ID (0 = sem cluster)
  3. fire_time_start                     : Timestamp em ms (0 = sem informação)
  4. fire_origin_zone                    : Binária (1 se 0 <= R_lapsetime <= 0.1; senão 0)
  5. fire_origin_point                   : Binária (1 se R_lapsetime == 0; senão 0)

REGRAS:
  - Trajetória válida: valores entre 0 e 1.
  - Fallback (2): área queimada sem trajetória estimada.
  - Zonas e pixels de origem são derivados da trajetória consolidada.
  - Em sobreposições, prevalece o mês de maior prioridade temporal.
  - A saída é limitada às cicatrizes mensais do Monitor do Fogo.

Exportação em Float64 para preservar os timestamps em milissegundos.
==============================================================================
"""

# ==============================================================================
# ⚙️ CONFIGURAÇÃO DAS BANDAS
# ==============================================================================

BANDAS_ORIGINAIS = ["R_lapsetime", "CLUSTER_ID", "system_time_start"]
BANDAS_FINAIS = ["fire_relative_propagation_lapestime", "fire_cluster_id", "fire_time_start"]
ORDEM_TEMPORAL = [0, -1, 1, -2, 2, -3, 3]
# Coleção intermediária e pasta de exportação definidas nas variáveis globais.
col_traj = ee.ImageCollection(gee_target_collection)


# ==============================================================================
# 🔥 BUSCA DAS IMAGENS ST-DBSCAN
# ==============================================================================

def get_traj_image(date):
    """Carrega trajetória mensal, renomeia bandas e trata NODATA (-9999)."""

    idx = date.format("YYYY_MM")
    col = col_traj.filter(ee.Filter.eq("system:index", idx))

    # Imagem vazia para meses sem raster ST-DBSCAN
    empty = (ee.Image.constant([0, 0, 0])
             .rename(BANDAS_FINAIS).toDouble()
             .updateMask(ee.Image.constant(0)))

    # Imagem de entrada (quando disponível)
    img = ee.Image(col.first())

    # 🔥 Trajetória: somente valores entre 0 e 1 são válidos
    r = img.select("R_lapsetime").toDouble()
    r = r.updateMask(r.gte(0).And(r.lte(1)))
    r = r.rename("fire_relative_propagation_lapestime")

    # 🧩 Cluster: NODATA recebe 0, sem invalidar trajetória
    cl = img.select("CLUSTER_ID").toDouble()
    cl = cl.updateMask(cl.neq(-9999)).unmask(0)
    cl = cl.rename("fire_cluster_id")

    # ⏱️ Tempo: banda de entrada system_time_start
    # Nome final: fire_time_start
    t = img.select("system_time_start").toDouble()
    t = t.updateMask(t.neq(-9999)).unmask(0)
    t = t.rename("fire_time_start")

    # Preserva trajetórias válidas, independentemente dos campos auxiliares
    selecionada = ee.Image.cat([r, cl, t])
    selecionada = selecionada.updateMask(r.mask()).toDouble()

    # Retorna uma imagem, mesmo quando não existe raster mensal
    return ee.Image(ee.Algorithms.If(
        col.size().gt(0), selecionada, empty
    ))


# ==============================================================================
# 🗺️ CONSOLIDAÇÃO MENSAL
# ==============================================================================

def consolidate_month(year, month):
    """Combina trajetórias ST-DBSCAN e área queimada com busca temporal ±3 meses."""

    # Cicatriz mensal do Monitor do Fogo.
    burn_img = ee.Image(
        f"{burned_asset_col}/brazil-{year}-{month}"
    ).selfMask()

    base_date = ee.Date.fromYMD(year, month, 1)

    # Composição inicial totalmente mascarada.
    composite = (ee.Image.constant([0, 0, 0])
                 .rename(BANDAS_FINAIS).toDouble()
                 .updateMask(ee.Image.constant(0)))

    # Busca temporal em cascata.
    # A ordem invertida garante prioridade ao mês central.
    for offset in reversed(ORDEM_TEMPORAL):
        target_date = base_date.advance(offset, "month")
        is_valid = target_date.get("year").gte(2019)

        traj = get_traj_image(target_date)

        composite = ee.Image(ee.Algorithms.If(
            is_valid,
            traj.unmask(composite, sameFootprint=False),
            composite
        ))

    # Fallback aplicado somente após combinar os meses disponíveis.
    fallback = (ee.Image.constant([2.0, 0.0, 0.0])
                .rename(BANDAS_FINAIS).toDouble())

    # Restringe os resultados à cicatriz mensal de área queimada.
    base_3bands = (composite.unmask(fallback, sameFootprint=False)
                   .updateMask(burn_img.gt(0)).toDouble())

    # 🟧 Zona de origem: valores de trajetória entre 0 e 0.1.
    r_lapse = base_3bands.select("fire_relative_propagation_lapestime")
    origin_zone = r_lapse.gte(0).And(r_lapse.lte(0.1))
    origin_zone = origin_zone.rename("fire_origin_zone").toDouble()

    # 📍 Pixels de origem: trajetória igual a zero.
    origin_point = r_lapse.eq(0)
    origin_point = origin_point.rename("fire_origin_point").toDouble()

    # Produto final com cinco bandas em Float64.
    return base_3bands.addBands([origin_zone, origin_point]).set({
        "year": year,
        "month": month,
        "system:index": f"{year}_{month:02d}",
        "system:time_start": base_date.millis()
    })


# ==============================================================================
# 🚀 EXPORTAÇÃO DIRETA EM LOTE PARA O GOOGLE EARTH ENGINE
# ==============================================================================

print(f"\n{C_YELLOW}{'=' * 65}")
print("🚀 PASSO 04 — CONSOLIDAÇÃO E EXPORTAÇÃO V1")
print(f"{'=' * 65}{C_RESET}")

tasks_dispatched = []

for year in range(export_start_year, export_end_year + 1):

    # Respeita os limites originais de ano e mês.
    m_start = export_start_month if year == export_start_year else 1
    m_end = export_end_month if year == export_end_year else 12

    for month in range(m_start, m_end + 1):

        asset_index = f"{year}_{month:02d}"
        target_asset_id = f"{gee_target_consolidated_collection}/{asset_index}"

        print(f"\n⏳ Exportando: {asset_index}")

        try:
            # Consolida as cinco bandas.
            imagem = consolidate_month(year, month)

            # Exportação para o asset mensal definitivo.
            task = ee.batch.Export.image.toAsset(
                image=imagem,
                description=f"IPAM_Fire_Monitor_FireTrajectory_Origin_{asset_index}",
                assetId=target_asset_id,
                scale=30,
                region=brasil_geom,
                maxPixels=1e13,
                pyramidingPolicy={
                    "fire_relative_propagation_lapestime": "mean",
                    "fire_cluster_id": "sample",
                    "fire_time_start": "sample",
                    "fire_origin_zone": "mode",
                    "fire_origin_point": "mode"
                }
            )

            task.start()
            tasks_dispatched.append((asset_index, task))

            print(f"🚀 {asset_index}: tarefa iniciada | Task ID: {task.id}")

        except Exception as e:
            print(f"❌ Erro na exportação de {asset_index}: {e}")

print(f"\n{C_GREEN}{'=' * 65}")
print(f"✅ PASSO 04: {len(tasks_dispatched)} tarefas iniciadas!")
print(f"{'=' * 65}{C_RESET}")


In [ ]:
import ee

ee.Initialize(project='mapbiomas-fire-485203')

# 🗑️ ImageCollection V1
colecao = 'projects/mapbiomas-workspace/FOGO/MONITORAMENTO/fire_monitor_trajetory_v1'

# Excluir todos os assets da coleção
for asset in ee.data.listAssets({'parent': colecao}).get('assets', []):
    ee.data.deleteAsset(asset['name'])
    print(f"✅ Excluído: {asset['name']}")

print('✅ Todas as imagens listadas foram excluídas!')